# Factorio Pyanodons Neutron Processing

Calculation for the *Neutron Capture* process for processing Plutonium by eliminating the waste products and processing them into wanted products.

This code is MIT-Licensed. See [`LICENSE.txt`](LICENSE.txt).

In [ ]:
import numpy as np
from scipy.optimize import linprog, dual_annealing

## *Neutron Capture* Process Definition

How many times must each *Neutron Capture* process run to eliminate all the unwanted products?

x0
: Neutron Absorption PU-240 (239/241 -> 240/242)

x1
: Neutron Absorption PU-238 (239/240 -> 238/242)

x2
: Neutron Absorption PU-241 (2x242 -> 241/239)

x3
: Neutron Absorption PU-239 (241/242 -> 239/240)

(All values are in percent, i.e. multiplied by 100)


In [ ]:
# Process definitions.
# Index: 238/239/240/241/242.
# Positive: output, Negative: input
# Output of the PU-Oxide procss. This is the input to the neutron capture process.
puox = np.array([2, 53, 25, 15, 50])
# Neutron capture processes
nc_240 = [0, -1, 1, -1, 1]
nc_238 = [1, -1, -1, 0, 1]
nc_241 = [0, 1, 0, 1, -2]
nc_239 = [0, 1, 1, -1, -1]
materials = ["PU-238", "PU-239", "PU-240", "PU-241", "PU-242"]
process_names = ["PU-240", "PU-238", "PU-241", "PU-239"]
# Production matrix
nc = np.array([nc_240, nc_238, nc_241, nc_239])
# Production matrix transposed
nc_t = nc.transpose()
# Duration of each process in seconds
duration = np.array([442, 202, 281, 552])


## *Neutron Capture* Process Output

PU-238 und PU-239 are useful.
PU-240, PU-241, PU-242 need to be removed.

## The Equation system

From the process definition we can derive the following system of equalities:

PU-238
:  2 +  0*x0 +  1*x1 + 0*x2 + 0*x3 = a, a >= 0

    0*x0 + -1*x1 + 0*x2 + 0*x3 <= 2

PU-239
: 53 + -1*x0 + -1*x1 + 1*x2 + 1*x3 = b, b >= 0

    1*x0 + 1*x1 + -1*x2 + -1*x3 <= 53

PU-240
: 25 + +1*x0 + -1*x1 + 0*x2 + 1*x3 = 0

    1*x0 + -1*x1 + 0*x2 + 1*x3 = -25

PU-241
: 15 + -1*x0 + 0*x1 + 1*x2 + -1*x3 = 0

    -1*x0 + 0*x1 + 1*x2 + -1*x3 = -15

PU-242
: 50 + 1*x0 + 1*x1 + -2*x2 + -1*x3 = 0 

    1*x0 + 1*x1 + -2*x2 + -1*x3 = -50


The equation system can be solved by using a linear programming minimization algorithm.

In [ ]:
# Build the equation system from the process definition.
pu238 = nc_t[0]
pu239 = nc_t[1]
pu240 = nc_t[2]
pu241 = nc_t[3]
pu242 = nc_t[4]
# print("pu238:", pu238)
# print("pu239:", pu239)
# print("pu240:", pu240)
# print("pu241:", pu241)
# print("pu242:", pu242)
a0 = pu238 * -1
a1 = pu239 * -1
A_ub = np.array([pu238 * -1, pu239 * -1])
b_ub = np.array([puox[0], puox[1]])
A_eq = np.array([pu240 * -1, pu241 * -1, pu242 * -1])
b_eq = np.array([puox[2], puox[3], puox[4]])
# Objective function: minimize the number of cycles
c = np.array([1, 1, 1, 1])
integrality = np.full_like(c, 1)  # All variables must be integers
# print("A_ub:", A_ub)
# print("b_ub:", b_ub)
# print("A_eq:", A_eq)
# print("b_eq:", b_eq)
res_cycles = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq)
print(f"Optimization result:\n{res_cycles}")
if res_cycles.success:
    for i in range(len(process_names)):
        print(f"{process_names[i]} neutron capture: {res_cycles.x[i]} times")
    cycles = res_cycles.x
    total_time = np.dot(cycles, duration)
    print(f"Total time: {total_time:3.0f} seconds")
else:
    print("Optimization failed:", res_cycles.message)
    raise ValueError("Exiting due to optimization failure.")

Since we know that there are no half cycles, we introduce a scale factor to make the number of cycles an integer value.

In [ ]:
# Multiply the number of cycles by a scale factor to get the total number of cycles needed to process all the PU-Oxide.
scale = 2  # Derived from the output of the optimization.
cycles *= scale
print(f"Total cycles needed to process all PU-Oxide: {cycles}")

# Verification

Simulate the proces using the number of cycles calculated in the previous step.

In [ ]:
# Run a simulation using the calculated number of cycles.
items = np.zeros_like(puox)
iteration = 0
done = False
while iteration < 10 and not done:
    iteration += 1
    # Add the output of the PU-Oxide process to the items.
    for count in range(scale):
        items += puox
    # Run the neutron capture processes for the calculated number of cycles.
    for i in range(4):
        print(f"Process {i}: {nc[i]} for {cycles[i]:3.0f} cycles")
        for j in range(int(cycles[i] + 0.5)):
            items += nc[i]
            if items[2] == 0 and items[3] == 0 and items[4] == 0:
                print(
                    f"Items after iteration {iteration}, process {i}, cycle {j}: {items}"
                )
                done = True
                break
        print(f"Iteration {iteration} - Items after process {i}: {items}")

if not done:
    print(f"Iteration failed: items after all processes: {items}")
else:
    print(f"Iteration succeeded: items after all processes: {items}")

### Calculate the optimum number of machines

Since the processes run in parallel, on multiple assemblers per process, we
want to know which number of machines is needed to run the process efficiently.

For a single machine, the runtime per process would be cycles * duration. Obviously, if we have *x*
machines, the runtime per process is cycles * duration / x.

We want to minimize the runtime per machine, while keeping the number of machines sensible.

The minimization problem is thus:

min cycles * duration / x

is

max x / (cycles * duration)

is

min - (x / (cycles * duration))


therefore:

c_n = - 1 / (cycles_n * duration_n)

In [ ]:
# Setup the optimization problem.
# Duration of each process
c = np.array(
    [cycles[1] * duration[1], cycles[2] * duration[2], cycles[3] * duration[3]]
)
print(f"Process durations (cycles*duration): {c}")


# Objective function: Maximum time needed per process.
def maxtime(x):
    mt = 0
    for i in range(len(x)):
        mt = max(mt, c[i] / x[i])
    # print(f"t = {mt}\nc = {c}\nx = {x}")
    return mt


max_machines = 6
lb = [1] * len(c)
ub = [max_machines] * len(c)
# Optimize the function
res_machines = dual_annealing(
    func=maxtime, bounds=list(zip(lb, ub)), x0=np.array([1, 1, 1]), maxiter=10000
)
print(f"Optimization result:\n{res_machines}")
print(f"Number of machines: {res_machines.x}")
machines = np.ceil(
    np.array([0, res_machines.x[0], res_machines.x[1], res_machines.x[2]])
)
print(f"Rounded number of machines: {machines}")

In [ ]:
# Alternative calculation
m1 = np.array([0, max_machines * c[0] / c[2], max_machines * c[1] / c[2], max_machines])
machines_manual = np.ceil(m1)
print(f"Manual calculation: {m1}")
print(f"Rounded: {machines_manual}")